author: ransell

# Attention Check Analysis


**How to run:** Use `Run All` from the top menu. Configure any file paths in the first code cell.

Loading raw data from connect and simplifying to just the columns needed

In [1]:
!pip install tabulate pandas

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)


In [3]:
import pandas as pd

input_csv = "VALUES-humanness_june_2025_TS_TT_July 14, 2025_17.06.csv"
df = pd.read_csv(input_csv)

# attn check cols 
columns_needed = ["ResponseId", "attn", "tt_res_10", "ts_res_10", "pairs"]
df = df[columns_needed]
df["pairs"] = pd.to_numeric(df["pairs"], errors="coerce")

output_csv = "data/attn_check_survey_data.csv"
df.to_csv(output_csv, index=False)

print(f"Cleaned data saved to {output_csv}")

Cleaned data saved to data/attn_check_survey_data.csv


Attention Checks Calculations

In [4]:
import pandas as pd

# Read the survey data
df = pd.read_csv("data/attn_check_survey_data.csv")
df["pairs"] = pd.to_numeric(df["pairs"], errors="coerce")

# Create boolean column for the original attention check
df["pass_check1"] = df["attn"].str.strip() == "9"

# Create boolean columns for the new attention checks (correct response: "Post A")
df["pass_check3"] = df["tt_res_10"].str.strip() == "1"
df["pass_check4"] = df["ts_res_10"].str.strip() == "1"

# Combined attention check results for all three checks 
df["pass_all"] = df["pass_check1"] & df["pass_check3"] & df["pass_check4"]
df["fail_all"] = (~df["pass_check1"]) & (~df["pass_check3"]) & (~df["pass_check4"])

grouped = df.groupby("pairs")
results = []

for pair_val, group in grouped:
    total_users = len(group)
    
    count_check1 = group["pass_check1"].sum()
    count_check3 = group["pass_check3"].sum()
    count_check4 = group["pass_check4"].sum()
    count_all = group["pass_all"].sum()
    
    count_fail_all = group["fail_all"].sum()
    
    # Calculate percentages 
    pct_check1 = (count_check1 / total_users) * 100 if total_users else 0
    pct_check3 = (count_check3 / total_users) * 100 if total_users else 0
    pct_check4 = (count_check4 / total_users) * 100 if total_users else 0
    pct_all = (count_all / total_users) * 100 if total_users else 0
    pct_fail_all = (count_fail_all / total_users) * 100 if total_users else 0

    results.append({
        "pairs": pair_val,
        "total_users": total_users,
        "pass_check1_obama": f"{count_check1} out of {total_users}",
        "pass_check1_pct": f"{pct_check1:.2f}%",
        "pass_check3_x_post": f"{count_check3} out of {total_users}",
        "pass_check3_pct": f"{pct_check3:.2f}%",
        "pass_check4_yt_post": f"{count_check4} out of {total_users}",
        "pass_check4_pct": f"{pct_check4:.2f}%",
        "pass_all": f"{count_all} out of {total_users}",
        "pass_all_pct": f"{pct_all:.2f}%",
        "fail_all": f"{count_fail_all} out of {total_users}",
        "fail_all_pct": f"{pct_fail_all:.2f}%"
    })

results_df = pd.DataFrame(results)

# Print 
print(results_df)

# Save 
output_csv = "data/attention_checks_analysis.csv"
results_df.to_csv(output_csv, index=False)
print(f"Analysis saved to {output_csv}")


   pairs  total_users pass_check1_obama pass_check1_pct pass_check3_x_post  \
0   15.0          608    589 out of 608          96.88%     591 out of 608   

  pass_check3_pct pass_check4_yt_post pass_check4_pct        pass_all  \
0          97.20%      582 out of 608          95.72%  563 out of 608   

  pass_all_pct      fail_all fail_all_pct  
0       92.60%  4 out of 608        0.66%  
Analysis saved to data/attention_checks_analysis.csv


In [5]:
from tabulate import tabulate

# nice output 
print(tabulate(results_df, headers='keys', tablefmt='psql'))

+----+---------+---------------+---------------------+-------------------+----------------------+-------------------+-----------------------+-------------------+----------------+----------------+--------------+----------------+
|    |   pairs |   total_users | pass_check1_obama   | pass_check1_pct   | pass_check3_x_post   | pass_check3_pct   | pass_check4_yt_post   | pass_check4_pct   | pass_all       | pass_all_pct   | fail_all     | fail_all_pct   |
|----+---------+---------------+---------------------+-------------------+----------------------+-------------------+-----------------------+-------------------+----------------+----------------+--------------+----------------|
|  0 |      15 |           608 | 589 out of 608      | 96.88%            | 591 out of 608       | 97.20%            | 582 out of 608        | 95.72%            | 563 out of 608 | 92.60%         | 4 out of 608 | 0.66%          |
+----+---------+---------------+---------------------+-------------------+--------------

In [6]:
# Filter out rows where not all attention checks passed (i.e. failed at least one)
failed_any_responseids = df.loc[~df["pass_all"], "ResponseId"]

# Convert the series into a DataFrame and save to CSV
failed_any_df = failed_any_responseids.to_frame()
failed_any_df.to_csv("data/failed_attn_ids.csv", index=False)

print("Saved ResponseIDs that failed at least one attention check to 'failed_attn_ids.csv'")


Saved ResponseIDs that failed at least one attention check to 'failed_attn_ids.csv'
